# alaz-solar-alert: Active Turkish Space Assets Discovery
This notebook serves as a verifiable record of all active Turkish space assets currently in orbit, used as the primary observation network for solar storm impact analysis.

In [3]:
import requests
import pandas as pd
import io

# --- SPACE-TRACK CREDENTIALS ---
#Enter your own username and password.
USERNAME = "Enter your own username"
PASSWORD = "Enter your own password" 

# Initialize a session for persistent authentication
session = requests.Session()
login_url = "https://www.space-track.org/ajaxauth/login"
login_data = {'identity': USERNAME, 'password': PASSWORD}

# Perform login
login_response = session.post(login_url, data=login_data)

if login_response.status_code == 200:
    print("Space-Track session established successfully. Ready to fetch data.")
else:
    print(f"Login failed! Status Code: {login_response.status_code}")

Space-Track session established successfully. Ready to fetch data.


In [4]:
# --- DYNAMIC QUERY FOR ACTIVE TURKISH ASSETS ---
# Filters: Country=TURK, Not Decayed (on-orbit), Updated within last 10 days
api_url = "https://www.space-track.org/basicspacedata/query/class/gp/COUNTRY_CODE/TURK/DECAY_DATE/null-val/EPOCH/%3Enow-10/orderby/NORAD_CAT_ID%20asc/emptyresult/show"

response = session.get(api_url)

if response.status_code == 200:
    # Convert JSON response to Pandas DataFrame
    df_assets = pd.DataFrame(response.json())
    
    # Orbit Classification Logic
    df_assets['MEAN_MOTION'] = df_assets['MEAN_MOTION'].astype(float)
    
    def classify_orbit(mm):
        if mm > 11:
            return "LEO (Low Earth Orbit)"
        elif 0.9 <= mm <= 1.1:
            return "GEO (Geostationary Orbit)"
        else:
            return "MEO / Other"

    df_assets['ORBIT_CATEGORY'] = df_assets['MEAN_MOTION'].apply(classify_orbit)
    
    # Display Summary Report
    print("--- ALAZSEZİ ORBITAL DISTRIBUTION REPORT ---")
    print(df_assets['ORBIT_CATEGORY'].value_counts())
    
    latest_asset = df_assets.sort_values(by='LAUNCH_DATE', ascending=False).iloc[0]
    print(f"\nLatest Deployed Asset: {latest_asset['OBJECT_NAME']} (Launched: {latest_asset['LAUNCH_DATE']})")
    print(f"Total Assets Tracked: {len(df_assets)}")
    
    # Show the definitive list of tracked assets
    display(df_assets[['NORAD_CAT_ID', 'OBJECT_NAME', 'ORBIT_CATEGORY', 'LAUNCH_DATE', 'EPOCH']])
else:
    print(f"Error fetching data! Status Code: {response.status_code}")

--- ALAZSEZİ ORBITAL DISTRIBUTION REPORT ---
ORBIT_CATEGORY
LEO (Low Earth Orbit)        30
GEO (Geostationary Orbit)     9
Name: count, dtype: int64

Latest Deployed Asset: CONNECTA IOT-15 (Launched: 2026-01-11)
Total Assets Tracked: 39


,NORAD_CAT_ID,OBJECT_NAME,ORBIT_CATEGORY,LAUNCH_DATE,EPOCH
0,23200,TURKSAT 1B,GEO (Geostationary Orbit),1994-08-10,2026-03-27T21:52:12.096768
1,23949,TURKSAT 1C,GEO (Geostationary Orbit),1996-07-09,2026-03-28T08:19:35.032800
2,26666,TURKSAT 2A,GEO (Geostationary Orbit),2001-01-10,2026-03-28T13:08:20.291712
3,27943,BILSAT 1,LEO (Low Earth Orbit),2003-09-27,2026-03-28T10:40:25.312224
4,33056,TURKSAT 3A,GEO (Geostationary Orbit),2008-06-12,2026-03-28T11:33:46.872864
5,35935,ITUPSAT 1,LEO (Low Earth Orbit),2009-09-23,2026-03-28T10:58:28.604928
6,37791,RASAT,LEO (Low Earth Orbit),2011-08-17,2026-03-28T13:30:15.670368
7,39030,GOKTURK 2,LEO (Low Earth Orbit),2012-12-18,2026-03-28T14:01:43.770432
8,39152,TURKSAT 3U,LEO (Low Earth Orbit),2013-04-26,2026-03-28T05:23:18.193632
9,39522,TURKSAT 4A,GEO (Geostationary Orbit),2014-02-14,2026-03-28T13:16:28.997760
